# CS 340 Project Two — Grazioso Salvare Dashboard
**Author:** Mohannad Abuimeish  
Interactive MongoDB dashboard built with Dash / JupyterDash. Uses the `AnimalShelter` CRUD module from Project One to filter the Austin Animal Center Outcomes data set by rescue-dog profile (Water, Mountain/Wilderness, Disaster/Individual Tracking) and to display results in a data table, a breed pie chart, and a geolocation map.

## Data Manipulation / Model
Connect to MongoDB via the Project One CRUD module and load the initial, unfiltered data set.

In [1]:
# Setup the Jupyter version of Dash
from dash import Dash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl

from dash import dcc
from dash import html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import the CRUD Python module built in Project One
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
username = "aacuser"
password = "SNHU1234!"

# Connect to the MongoDB "aac" database via the CRUD module
db = AnimalShelter(username, password)

# Retrieve all documents to populate the initial, unfiltered dashboard view
df = pd.DataFrame.from_records(db.read({}))

# MongoDB returns an '_id' column of type ObjectId, which the DataTable
# component cannot render, so it is dropped here
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)


## Rescue-Type Filter Queries
Queries built from the Rescue Type and Preferred Dog Breeds table in the Dashboard Specifications Document.

In [2]:
##############################################
# Rescue-Type Filter Queries
# Built from the Rescue Type and Preferred Dog
# Breeds Table in the Dashboard Specifications
# Document. Breed is matched with a case-
# insensitive regex since the dataset stores
# breeds like "Labrador Retriever Mix" rather
# than an exact match.
##############################################

WATER_QUERY = {
    "animal_type": "Dog",
    "breed": {
        "$regex": "Labrador Retriever Mix|Chesapeake Bay Retriever|Newfoundland",
        "$options": "i"
    },
    "sex_upon_outcome": "Intact Female",
    "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
}

MOUNTAIN_QUERY = {
    "animal_type": "Dog",
    "breed": {
        "$regex": "German Shepherd|Alaskan Malamute|Old English Sheepdog|Siberian Husky|Rottweiler",
        "$options": "i"
    },
    "sex_upon_outcome": "Intact Male",
    "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
}

DISASTER_QUERY = {
    "animal_type": "Dog",
    "breed": {
        "$regex": "Doberman Pinscher|German Shepherd|Golden Retriever|Bloodhound|Rottweiler",
        "$options": "i"
    },
    "sex_upon_outcome": "Intact Male",
    "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
}

RESET_QUERY = {}


def get_query_for_filter(filter_type):
    """Map a filter-type radio button value to its MongoDB query."""
    if filter_type == 'water':
        return WATER_QUERY
    elif filter_type == 'mountain':
        return MOUNTAIN_QUERY
    elif filter_type == 'disaster':
        return DISASTER_QUERY
    else:
        return RESET_QUERY


## Dashboard Layout / View

In [3]:
#########################
# Dashboard Layout / View
#########################
app = Dash(__name__, requests_pathname_prefix='/proxy/8050/')
# Grazioso Salvare logo, wrapped in an anchor tag to the client's home page
# as required by the branding specifications. Place Grazioso_Salvare_Logo.png
# in the same directory as this notebook (it is preloaded in Codio).
image_filename = 'Grazioso_Salvare_Logo.png'
try:
    encoded_image = base64.b64encode(open(image_filename, 'rb').read())
    logo_element = html.A(
        href='https://www.snhu.edu',
        target='_blank',
        children=html.Img(
            src='data:image/png;base64,{}'.format(encoded_image.decode()),
            style={'height': '120px'}
        )
    )
except FileNotFoundError:
    # Allows the dashboard to still run if the logo file is not present
    # in the working directory; replace with the real logo before submitting.
    logo_element = html.A(
        'Grazioso Salvare',
        href='https://www.snhu.edu',
        target='_blank'
    )

app.layout = html.Div([
    html.Center(logo_element),
    html.Center(html.B(html.H1('SNHU CS-340 Dashboard'))),
    html.Center(html.P('Unique Identifier: MoeAbuimeish_CS340')),
    html.Hr(),

    # Interactive filter options required by Grazioso Salvare
    html.Div(
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain'},
                {'label': 'Disaster or Individual Tracking', 'value': 'disaster'},
                {'label': 'Reset', 'value': 'reset'}
            ],
            value='reset',
            labelStyle={'display': 'inline-block', 'margin-right': '20px'}
        )
    ),
    html.Hr(),

    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        page_size=10,
        page_action='native',
        sort_action='native',
        filter_action='native',
        row_selectable='single',
        selected_rows=[0],
        style_table={'overflowX': 'auto'},
        style_cell={'textAlign': 'left', 'minWidth': '100px'}
    ),
    html.Br(),
    html.Hr(),

    # Chart and geolocation map, displayed side-by-side
    html.Div(
        className='row',
        style={'display': 'flex'},
        children=[
            html.Div(
                id='graph-id',
                className='col s12 m6',
            ),
            html.Div(
                id='map-id',
                className='col s12 m6',
            )
        ]
    )
])


## Interaction Between Components / Controller

In [4]:
#############################################
# Interaction Between Components / Controller
#############################################

# Updates the data table whenever a new filter option is selected.
# The controller (this callback) uses the CRUD Python module to run the
# corresponding query against MongoDB and returns fresh data to the view.
@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
    query = get_query_for_filter(filter_type)
    filtered_df = pd.DataFrame.from_records(db.read(query))

    if not filtered_df.empty and '_id' in filtered_df.columns:
        filtered_df.drop(columns=['_id'], inplace=True)

    return filtered_df.to_dict('records')


# Displays a pie chart of breed distribution based on whatever data is
# currently shown in the data table (i.e., it responds to the active filter).
@app.callback(
    Output('graph-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data')]
)
def update_graphs(viewData):
    if viewData is None:
        return []

    dff = pd.DataFrame.from_dict(viewData)
    if dff.empty:
        return [html.P('No matching animals for this filter.')]

    return [
        dcc.Graph(
            figure=px.pie(dff, names='breed', title='Preferred Animals by Breed')
        )
    ]


# Highlights a column when the user selects it in the data table.
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if not selected_columns:
        return []
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# Updates the geolocation map to show the currently selected animal from
# the data table. Falls back to the first row if nothing is explicitly
# selected, and clears the map if the filter returns no results.
@app.callback(
    Output('map-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data'),
     Input('datatable-id', 'derived_virtual_selected_rows')]
)
def update_map(viewData, index):
    if not viewData:
        return []

    dff = pd.DataFrame.from_dict(viewData)
    if dff.empty:
        return [html.P('No matching animals for this filter.')]

    row = index[0] if index else 0
    row = min(row, len(dff) - 1)

    # Austin, TX is used as the default map center
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'},
               center=[30.75, -97.48], zoom=10, children=[
                   dl.TileLayer(id="base-layer-id"),
                   dl.Marker(
                       position=[dff.iloc[row]['location_lat'], dff.iloc[row]['location_long']],
                       children=[
                           dl.Tooltip(dff.iloc[row]['breed']),
                           dl.Popup([
                               html.H1("Animal Name"),
                               html.P(dff.iloc[row]['name'])
                           ])
                       ]
                   )
               ])
    ]


## Run the Dashboard

In [ ]:
app.run(debug=False, host='0.0.0.0', port=8050)

Dash is running on http://0.0.0.0:8050/proxy/8050/

 * Serving Flask app '__main__' (lazy loading)
 * Environment: production
   Use a production WSGI server instead.
 * Debug mode: off


 * Running on all addresses.
 * Running on http://10.15.222.85:8050/ (Press CTRL+C to quit)
127.0.0.1 - - [13/Aug/2026 15:49:36] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [13/Aug/2026 15:49:36] "GET /_dash-dependencies HTTP/1.1" 200 -
127.0.0.1 - - [13/Aug/2026 15:49:37] "GET /_dash-layout HTTP/1.1" 200 -
127.0.0.1 - - [13/Aug/2026 15:49:37] "GET /_dash-component-suites/dash/dash_table/async-table.js HTTP/1.1" 304 -
127.0.0.1 - - [13/Aug/2026 15:49:37] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [13/Aug/2026 15:49:37] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [13/Aug/2026 15:49:37] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [13/Aug/2026 15:49:37] "GET /_dash-component-suites/dash/dash_table/async-highlight.js HTTP/1.1" 304 -
127.0.0.1 - - [13/Aug/2026 15:49:38] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [13/Aug/2026 15:49:45] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [13/Aug/2026 15:49:45] "POST /_dash-upd